<a href="https://colab.research.google.com/github/jdmartinev/CVBootcampMCDA/blob/main/referencias/ref_02_tensor_operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Referencia rápida: Operaciones con tensores en PyTorch

> Tenla abierta mientras trabajas en los TODOs del workshop. Cada sección es ejecutable.

| Sección | Cuándo consultarla |
|---------|-------------------|
| 1. Shapes | Antes de cualquier operación — entender qué tiene cada variable |
| 2. Producto matricial | TODO 3, TODO 4 — `compute_similarity_matrix`, `compute_scores` |
| 3. Normalización L2 | TODO 1, TODO 2 — al final de `get_*_embeddings` |
| 4. `topk` | TODO 6, TODO 7 — obtener las top-k imágenes |
| 5. `argsort` | TODO C — Reciprocal Rank Fusion |
| 6. Indexar con tensor | TODO 6, TODO 7 — recuperar imágenes por índice |
| 7. Concatenar | TODO 5 — `build_image_index` en batches |
| 8. Fusión lineal | TODO B — `search_by_reference` |
| 9. Verificaciones | Cuando algo da un shape o valor inesperado |

In [ ]:
import torch
import torch.nn.functional as F

print(f"PyTorch: {torch.__version__}")

---
## 1. Shapes y dimensiones

Entender el shape de un tensor es lo primero antes de cualquier operación.

**Shapes más comunes en este workshop:**

| Variable | Shape | Significado |
|----------|-------|-------------|
| `text_embs` | `(N, 512)` | N textos codificados |
| `image_embs` | `(N, 512)` | N imágenes codificadas |
| `query_emb` | `(1, 512)` | un único query |
| `scores` | `(N,)` | un score por imagen del corpus |
| `similarity_matrix` | `(N, N)` | similitud entre N imágenes y N textos |

In [ ]:
t = torch.randn(3, 512)  # 3 embeddings de 512 dimensiones

print(f"t.shape:     {t.shape}")
print(f"t.shape[0]:  {t.shape[0]}  → número de vectores (batch size)")
print(f"t.shape[-1]: {t.shape[-1]}  → dimensión del embedding (última dim)")
print(f"t.ndim:      {t.ndim}       → número de dimensiones")
print()

# La diferencia entre (N,) y (N, 1) y (1, N)
a = torch.randn(5)       # shape (5,)   — vector 1D
b = torch.randn(5, 1)    # shape (5, 1) — columna
c = torch.randn(1, 5)    # shape (1, 5) — fila
print(f"(5,)   → {a.shape}")
print(f"(5, 1) → {b.shape}")
print(f"(1, 5) → {c.shape}")
print()
print(f"squeeze de (1, 5): {c.squeeze().shape}  ← elimina dimensiones de tamaño 1")

---
## 2. Producto punto y multiplicación matricial

Con vectores **normalizados L2**, la multiplicación matricial calcula similitud coseno para todos los pares a la vez.

> Si `a` y `B` están normalizados (norma = 1), entonces `(a @ B.T)[i] = cos(a, B[i])`.

In [ ]:
# Query vs corpus
query  = F.normalize(torch.randn(1, 512), p=2, dim=-1)   # (1, 512)
corpus = F.normalize(torch.randn(800, 512), p=2, dim=-1) # (800, 512)

# (1, 512) @ (512, 800) → (1, 800)
scores = query @ corpus.T
print(f"query @ corpus.T  →  shape: {scores.shape}")

# squeeze elimina la dimensión de tamaño 1
scores = scores.squeeze()
print(f"después de squeeze  →  shape: {scores.shape}")
print(f"rango de valores:  [{scores.min():.3f}, {scores.max():.3f}]  (siempre en [-1, 1])")

In [ ]:
# Matriz N×N — similitud entre N imágenes y N textos
img_embs  = F.normalize(torch.randn(15, 512), p=2, dim=-1)  # (15, 512)
text_embs = F.normalize(torch.randn(15, 512), p=2, dim=-1)  # (15, 512)

S = img_embs @ text_embs.T   # (15, 512) @ (512, 15) → (15, 15)
print(f"Matriz de similitud S:  {S.shape}")
print(f"S[i,j] = sim(imagen_i, texto_j)")
print(f"Diagonal (pares correctos): media = {S.diagonal().mean():.3f}")
print(f"Off-diagonal (aleatorios):  media = {(S.sum() - S.diagonal().sum()) / (15*14):.3f}")

---
## 3. Normalización L2

Divide cada vector por su norma para que quede en la esfera unitaria (norma = 1).

- `p=2` → norma L2 (euclidiana)  
- `dim=-1` → normaliza sobre la **última** dimensión (los 512 features), no sobre el batch

In [ ]:
# Ejemplo con números simples
v = torch.tensor([[3.0, 4.0]])   # norma = sqrt(9+16) = 5
v_norm = F.normalize(v, p=2, dim=-1)

print(f"Original:   {v}  →  norma = {v.norm():.1f}")
print(f"Normalizado: {v_norm}  →  norma = {v_norm.norm():.1f}")
print()

# Batch de vectores: cada fila se normaliza independientemente
batch = torch.randn(5, 512)
print(f"Normas antes:  {batch.norm(dim=-1).round(decimals=2)}")

batch_norm = F.normalize(batch, p=2, dim=-1)
print(f"Normas después: {batch_norm.norm(dim=-1).round(decimals=2)}")

---
## 4. `torch.topk` — los K valores más altos

Retorna los K valores más altos **y sus índices originales**, ordenados de mayor a menor.

In [ ]:
scores = torch.tensor([0.1, 0.8, 0.3, 0.95, 0.6])
print(f"Scores: {scores.tolist()}")
print()

result = torch.topk(scores, k=3)
print(f"top-3 values:  {result.values.tolist()}   ← ordenados de mayor a menor")
print(f"top-3 indices: {result.indices.tolist()}  ← índices en el tensor original")
print()

# Uso típico en el workshop
top_k = 10
scores_corpus = F.normalize(torch.randn(1, 512), p=2, dim=-1) @ \
                F.normalize(torch.randn(800, 512), p=2, dim=-1).T
scores_corpus = scores_corpus.squeeze()

top = torch.topk(scores_corpus, k=top_k)
top_indices = top.indices   # shape (10,) — índices de las 10 imágenes más similares
top_scores  = top.values    # shape (10,) — sus scores
print(f"top_indices shape: {top_indices.shape}")
print(f"top_scores shape:  {top_scores.shape}")
print(f"Mejor score: {top_scores[0]:.4f}  |  Peor de los top-10: {top_scores[-1]:.4f}")

---
## 5. `argsort` — ranking completo

> **`topk` vs `argsort`:** usa `topk` para obtener solo los K mejores (más eficiente).  
> Usa `argsort` cuando necesitas el **ranking completo** — por ejemplo para Reciprocal Rank Fusion.

In [ ]:
scores = torch.tensor([0.1, 0.8, 0.3, 0.95, 0.6])
print(f"Scores: {scores.tolist()}")
print()

# Ascendente: índice 0 = el más bajo
idx_asc = scores.argsort()
print(f"argsort() ascendente:  {idx_asc.tolist()}")
print(f"  → scores ordenados:  {scores[idx_asc].tolist()}")
print()

# Descendente: índice 0 = el más alto
idx_desc = scores.argsort(descending=True)
print(f"argsort(descending=True): {idx_desc.tolist()}")
print(f"  → scores ordenados:     {scores[idx_desc].tolist()}")
print()

# Para RRF: necesitas saber el rank de cada imagen (posición en el ranking)
# rank[i] = ¿en qué posición está la imagen i?
N = len(scores)
rank = torch.zeros(N)
rank[idx_desc] = torch.arange(1, N + 1).float()  # rank 1 = mejor
print(f"Ranks (1=mejor): {rank.tolist()}")
print(f"  → imagen 3 (score=0.95) tiene rank {rank[3].int().item()}")

---
## 6. Indexar un tensor con otro tensor

Dado un tensor de índices, seleccionar esas filas del corpus.

In [ ]:
# Corpus de embeddings
image_embs = torch.randn(800, 512)
top_indices = torch.tensor([3, 45, 12, 7, 99])

# Seleccionar las filas correspondientes
top_embs = image_embs[top_indices]
print(f"image_embs shape:  {image_embs.shape}")
print(f"top_indices:       {top_indices.tolist()}")
print(f"top_embs shape:    {top_embs.shape}   ← 5 filas seleccionadas")
print()

# Lo mismo con listas de Python (necesitas .tolist())
corpus_captions = [f"caption_{i}" for i in range(800)]
top_captions = [corpus_captions[i] for i in top_indices.tolist()]
print(f"Top captions: {top_captions}")
print()
print("⚠️  .tolist() es necesario para indexar listas de Python con un tensor")

---
## 7. Concatenar tensores

Patrón para construir el índice procesando el corpus en batches.

In [ ]:
# Simular procesamiento en batches
batch1 = torch.randn(32, 512)
batch2 = torch.randn(32, 512)
batch3 = torch.randn(16, 512)  # último batch puede ser más pequeño

# Concatenar a lo largo de la dimensión 0 (filas)
full = torch.cat([batch1, batch2, batch3], dim=0)
print(f"batch1: {batch1.shape}")
print(f"batch2: {batch2.shape}")
print(f"batch3: {batch3.shape}")
print(f"cat →   {full.shape}  ← 32+32+16 = 80 filas")
print()

# Patrón del TODO 5 (build_image_index)
N_IMAGES = 100
BATCH_SIZE = 32
fake_images = [None] * N_IMAGES  # placeholder

all_embeddings = []
for i in range(0, N_IMAGES, BATCH_SIZE):
    batch = fake_images[i : i + BATCH_SIZE]
    # en el TODO real: embs = get_image_embeddings(batch, model, processor, device)
    embs = F.normalize(torch.randn(len(batch), 512), p=2, dim=-1)
    all_embeddings.append(embs)
    print(f"  batch {i//BATCH_SIZE + 1}: {embs.shape}")

index = torch.cat(all_embeddings, dim=0)
print(f"Índice final: {index.shape}")

---
## 8. Operaciones elemento a elemento

La fusión lineal del TODO B: combinar scores de texto e imagen con peso `alpha`.

In [ ]:
# Scores de dos fuentes para un corpus de 5 imágenes
score_text  = torch.tensor([0.9, 0.2, 0.6, 0.1, 0.4])
score_image = torch.tensor([0.3, 0.8, 0.5, 0.7, 0.2])

print(f"Scores texto:  {score_text.tolist()}")
print(f"Scores imagen: {score_image.tolist()}")
print()

# Explorar distintos valores de alpha
for alpha in [1.0, 0.7, 0.5, 0.0]:
    combined = alpha * score_text + (1 - alpha) * score_image
    best_idx = combined.argmax().item()
    print(f"alpha={alpha:.1f} → scores={[round(x,2) for x in combined.tolist()]}  mejor: img[{best_idx}]")

print()
print("alpha=1.0 → texto puro (ignora imagen)")
print("alpha=0.0 → imagen pura (ignora texto, mismo resultado para todos los queries)")
print("alpha∈(0,1) → híbrido — la imagen actúa como prior temático")

---
## 9. Verificaciones útiles durante el desarrollo

Celdas de diagnóstico para cuando algo da un shape o valor inesperado.

In [ ]:
# Simular outputs de tus funciones
N = 50
embeddings = F.normalize(torch.randn(N, 512), p=2, dim=-1)
scores = (F.normalize(torch.randn(1, 512), p=2, dim=-1) @ embeddings.T).squeeze()

# ── Verificar shape ────────────────────────────────────────────────────────────
assert embeddings.shape == (N, 512), f"Esperaba ({N}, 512), obtuve {embeddings.shape}"
print(f"✅ Shape correcto: {embeddings.shape}")

# ── Verificar normalización L2 ─────────────────────────────────────────────────
norms = embeddings.norm(dim=-1)
assert torch.allclose(norms, torch.ones(N), atol=1e-5), f"No normalizado: {norms[:5]}"
print(f"✅ Normalizado L2: normas ∈ [{norms.min():.6f}, {norms.max():.6f}]")

# ── Verificar rango de scores ──────────────────────────────────────────────────
assert scores.min() >= -1.01 and scores.max() <= 1.01, \
    f"Scores fuera de rango [-1,1]: [{scores.min():.3f}, {scores.max():.3f}]"
print(f"✅ Scores en rango [-1, 1]: [{scores.min():.3f}, {scores.max():.3f}]")

# ── Ver las primeras filas de un tensor grande ─────────────────────────────────
print(f"\nPrimeras 3 filas de embeddings:")
print(embeddings[:3])     # primeras 3 filas
print(f"Shape: {embeddings.shape}")

---
## Flujo completo del workshop

Todos los shapes en cada paso del pipeline.

In [ ]:
# Simulación del pipeline completo con shapes reales
print("── TEXTO ─────────────────────────────────────────────────────────")
texts = ["a dog playing", "a cat sleeping"]  # N=2 textos
# processor(text=texts) → token tensors
# model.get_text_features(**inputs)
text_embs_raw = torch.randn(2, 512)          # shape (N, 512) — sin normalizar
text_embs = F.normalize(text_embs_raw, p=2, dim=-1)
print(f"  processor + get_text_features → {text_embs_raw.shape}")
print(f"  F.normalize(p=2, dim=-1)      → {text_embs.shape}  normas={text_embs.norm(dim=-1).tolist()}")

print()
print("── IMAGEN ────────────────────────────────────────────────────────")
# processor(images=images) → pixel tensors
# model.get_image_features(**inputs)
img_embs_raw = torch.randn(800, 512)         # shape (N, 512) — sin normalizar
img_embs = F.normalize(img_embs_raw, p=2, dim=-1)
print(f"  processor + get_image_features → {img_embs_raw.shape}")
print(f"  F.normalize(p=2, dim=-1)       → {img_embs.shape}")

print()
print("── BÚSQUEDA ──────────────────────────────────────────────────────")
query_emb = text_embs[:1]                    # shape (1, 512)
print(f"  query_emb:         {query_emb.shape}")

scores = (query_emb @ img_embs.T).squeeze()  # (1,512)@(512,800) → (1,800) → (800,)
print(f"  query @ corpus.T:  {query_emb.shape} @ {img_embs.T.shape} → {scores.shape}")

top = torch.topk(scores, k=5)
print(f"  torch.topk(k=5):   indices={top.indices.shape}  values={top.values.shape}")
print(f"  top-5 índices: {top.indices.tolist()}")
print(f"  top-5 scores:  {[round(x, 3) for x in top.values.tolist()]}")